### config

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import config.ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("fact_rides",7)
spark.getActiveSession()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
25/03/19 13:56:09 WARN Utils: Your hostname, system resolves to a loopback address: 127.0.1.1; using 10.140.67.105 instead (on interface wlp170s0)
25/03/19 13:56:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/souls/.ivy2/cache
The jars for the packages stored in: /home/souls/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f07e70c2-f6f3-4d79-b8c7-444987ca3901;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/souls/Projects/data4/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found org.elasticsearch#elasticsearch-spark-30_2.12;8.15.2 in central
	found org.scala-lang#scala-reflect;2.

# EXTRACT

I tried using the spark API first, but there are some limitations on their joining of columns.

In [38]:
# EXTRACT rides:
# rides_table_SQL = '(SELECT * FROM rides) as rides_table'
# df_rides = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", rides_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_rides.printSchema()
#df_rides.show()
#df_rides.count()
# ----

#EXTRACT bike_type through bike_lot through vehicle:
# Rename to avoid duplicate column names
# Corrected column name
# vehicle_table_SQL = """
# (
#     SELECT
#         v.*,
#         l.bikelotid AS bikelotid_bikelots,
#         l.deliverydate,
#         l.biketypeid
#     FROM vehicles v
#     LEFT JOIN bikelots l ON v.bikelotid = l.bikelotid
# ) AS vehicle_table
# """
#
# df_vehicles = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", vehicle_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "vehicleid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_vehicles.show()
# ------

#EXTRACT users through subscriptionid through subscription
# user_table_SQL = """
# (
#     SELECT
#         userid AS userid_subscriptions,
#         subscriptionid AS subscriptionid_subscriptions
#     FROM subscriptions s
# ) AS user_table
# """
#
# df_users = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", user_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "userid_subscriptions") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_users.show()

# ------

#EXTRACT date
# df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
# df_dates.show()
# df_dates.createOrReplaceTempView("dates")

# ------

#EXTRACT weather: TBD.



Trying to do it in one SQL query:

In [37]:
# EXTRACTING ALL IN ONE GO:
# load dates from deltatable:
df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
#df_dates.show()
df_dates.createOrReplaceTempView("dates")

# write the SQL string needed:
SQL = """(
    SELECT
    r.*,
    v.vehicleid AS vehicleid_vehicles,
    v.bikelotid AS bikelotid_vehicles,
    b.bikelotid AS bikelotid_bikelots,
    b.biketypeid AS biketypeid_bikelots,
    s.subscriptionid AS subscriptionid_subscriptions,
    s.userid AS userid_subscriptions
        FROM rides r
            LEFT JOIN vehicles v ON r.vehicleid = v.vehicleid
            LEFT JOIN bikelots b ON v.bikelotid = b.bikelotid
            LEFT JOIN subscriptions s ON r.subscriptionid = s.subscriptionid

) as rides_table
"""

df_rides_full = spark.read.format("jdbc")\
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", SQL) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()

#df_rides_full.show()
# WEATHER would be done separately


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|vehicleid_vehicles|bikelotid_vehicles|bikelotid_bikelots|biketypeid_bikelots|subscriptionid_subscriptions|userid_subscriptions|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|               844|                 3|                 3|                  1|               

# TRANSFORM

In [ ]:
dataframe_rides = df_rides_full \
    .withColumnRenamed() \ # Add more

# LOAD

In [ ]:
spark.stop()